In [ ]:
pip
install
tree_sitter
tree_sitter_languages
tree - sitter - go

In [72]:
from __future__ import annotations  # ← оставляем самым первым

import json
import os
import tempfile
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Iterable, List, Optional, Set, Tuple, Union, Dict

# глушим лишние треды от нативных либ до их импорта
os.environ.setdefault("OMP_NUM_THREADS", "1")

from tree_sitter import Parser, Query  # Language напрямую не нужен с tsgo
import tree_sitter_go as tsgo          # используем tsgo.language()

In [73]:
# Если не включён в ноутбуке ранее:
# from __future__ import annotations

from dataclasses import dataclass, field
from typing import List, Optional, Tuple

# ──────────────────────────────────────────────────────────────────────────────
# Go models
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class GoImport:
    path: str
    alias: Optional[str] = None


@dataclass
class GoField:
    """Поле структуры Go. Если type == 'struct { ... }', то fields содержит вложенные поля."""
    name: str                         # Имя поля (или тип для embedded)
    type: str                         # Текст типа как в коде (вкл. inline: "struct {...}")
    tag: Optional[str] = None         # Тег (без кавычек), если есть
    embedded: bool = False            # Встраиваемое поле (embedded)
    fields: Optional[List["GoField"]] = None  # Для inline-struct; None если не struct


@dataclass
class GoType:
    """Тип верхнего уровня: struct/interface/alias/other."""
    name: str
    kind: str                                   # struct|interface|alias|other
    fields: Optional[List[Tuple[str, str]]] = None   # [(name, type)] для struct/alias
    methods: Optional[List[str]] = None              # интерфейсы: сигнатуры; alias: [target]
    line: Optional[int] = None


@dataclass
class GoParam:
    """Параметр функции/метода Go."""
    name: Optional[str]          # может отсутствовать (анонимный)
    type: str                    # int, *T, pkg.X, []byte, map[string]int, chan T, ...
    variadic: bool = False       # ...T


@dataclass
class GoFunc:
    """Функция или метод Go."""
    name: str
    receiver: Optional[str] = None   # "T" или "*T" для методов
    exported: bool = False
    params: List[GoParam] = field(default_factory=list)   # входные параметры
    results: List[str] = field(default_factory=list)      # выходные типы
    doc: Optional[str] = None
    line: Optional[int] = None


@dataclass
class GoFileMeta:
    """Метаданные по одному .go файлу."""
    path: str
    package: Optional[str]
    imports: List[GoImport] = field(default_factory=list)
    types: List[GoType] = field(default_factory=list)
    funcs: List[GoFunc] = field(default_factory=list)


# ──────────────────────────────────────────────────────────────────────────────
# Proto models
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class ProtoField:
    """Поле message в .proto."""
    name: str
    type: str
    number: int
    label: Optional[str] = None          # optional|repeated (proto3), required (proto2)
    default: Optional[str] = None        # текстовое значение по умолчанию (если указано)


@dataclass
class ProtoMessage:
    """Message в .proto."""
    name: str
    fields: List[ProtoField] = field(default_factory=list)


@dataclass
class ProtoEnumValue:
    name: str
    number: int


@dataclass
class ProtoEnum:
    """Enum в .proto."""
    name: str
    values: List[ProtoEnumValue] = field(default_factory=list)


@dataclass
class ProtoServiceMethod:
    """RPC-метод сервиса."""
    name: str
    input_type: str                       # имя message запроса
    output_type: str                      # имя message ответа
    client_streaming: bool = False
    server_streaming: bool = False


@dataclass
class ProtoService:
    """Service в .proto."""
    name: str
    methods: List[ProtoServiceMethod] = field(default_factory=list)


@dataclass
class ProtoFileMeta:
    """Метаданные по одному .proto файлу."""
    path: str
    package: Optional[str] = None
    messages: List[ProtoMessage] = field(default_factory=list)
    enums: List[ProtoEnum] = field(default_factory=list)
    services: List[ProtoService] = field(default_factory=list)

In [74]:
# ──────────────────────────────────────────────────────────────────────────────
# Инициализация tree-sitter-go
# ──────────────────────────────────────────────────────────────────────────────
def _load_go_language() -> Language:
    """
    Загружает Language для Go строго из tree-sitter-go.

    Способы:
      1) Среда: TREE_SITTER_GO_SO=/abs/path/to/go.so  (готовая сборка)
      2) Среда: TREE_SITTER_GO_GRAMMAR=/abs/path/to/tree-sitter-go  (автосборка)
         → соберём .so во временную дирекцию (нужен gcc/clang).

    Исключение — если ничего не найдено.
    """
    so_path = os.getenv("TREE_SITTER_GO_SO")
    if so_path:
        return Language(so_path, "go")

    grammar_dir = os.getenv("TREE_SITTER_GO_GRAMMAR")
    if grammar_dir:
        # Сборка .so на лету
        out_dir = Path(tempfile.gettempdir()) / "tsgo_build"
        out_dir.mkdir(parents=True, exist_ok=True)
        lib_path = str(out_dir / "tree_sitter_go.so")
        # Важно: Language.build_library перезапишет, если файл уже есть
        Language.build_library(
            # итоговая общая библиотека
            lib_path,
            # пути до репозиториев грамматик
            [grammar_dir],
        )
        return Language(lib_path, "go")

    raise RuntimeError(
        "Не найден tree-sitter-go. "
        "Установите один из вариантов:\n"
        "  • export TREE_SITTER_GO_SO=/abs/path/to/tree_sitter_go.so\n"
        "  • export TREE_SITTER_GO_GRAMMAR=/abs/path/to/tree-sitter-go   # исходники грамматики\n"
        "Пример грамматики: https://github.com/tree-sitter/tree-sitter-go"
    )


def make_go_parser() -> Parser:
    p = Parser(Language(tsgo.language()))
    return p


In [75]:
# ──────────────────────────────────────────────────────────────────────────────
# Обход проекта
# ──────────────────────────────────────────────────────────────────────────────

from pathlib import Path
from typing import Iterable, Optional, Set, Union, Iterable

GO_FILE_EXTS = frozenset({".go"})
GO_AUX_FILES = frozenset({"go.mod", "go.sum"})


def _is_ignored_dir(path: Path, ignore_dirs: Set[str]) -> bool:
    nm = path.name
    return (
        nm in ignore_dirs
        or (nm.startswith(".") and nm not in {".", ".."})
        or nm in {"vendor", "node_modules", "dist", "build", "out"}
    )


def iter_go_files(PROJECT_PATH: Union[str, Path], IGNORE_DIRS: Iterable[str]) -> Iterable[Path]:
    """
    Ищет .go и служебные файлы (go.mod/go.sum), пропуская игнорируемые директории.
    Без изменения логики: остаёмся на rglob, но ускоряем проверку родителей.
    """
    root = Path(PROJECT_PATH).resolve()
    ignore = {d.strip("/").strip() for d in IGNORE_DIRS if d}

    for p in root.rglob("*"):
        if p.is_dir():
            continue
        # Быстрые отсеки до проверки родителей
        if p.suffix not in GO_FILE_EXTS and p.name not in GO_AUX_FILES:
            continue
        # Проверяем, не попадает ли любой сегмент пути в игнор
        # (эквивалентно твоей проверке родителей, но без обхода всех parent до root.parent)
        if any(_is_ignored_dir(Path(part), ignore) for part in p.parts):
            # Примечание: Path(part) здесь используется только для name,
            # можно заменить на простой объект с .name = part, но так чище.
            continue
        yield p


# ──────────────────────────────────────────────────────────────────────────────
# Парсинг одного файла
# ──────────────────────────────────────────────────────────────────────────────

def _text(src: bytes, node) -> str:
    # Унифицируем извлечение текста: один helper вместо двух
    return src[node.start_byte:node.end_byte].decode("utf-8", errors="ignore")


def _strip_quotes(s: str) -> str:
    s = s.strip()
    if len(s) >= 2 and s[0] == s[-1] and s[0] in {'"', '`'}:
        return s[1:-1]
    return s


def _string_literal_text(src: bytes, node) -> Optional[str]:
    if node.type in ("interpreted_string_literal", "raw_string_literal"):
        return _strip_quotes(_text(src, node))
    return None


def _collect_leading_comments(src: bytes, node) -> Optional[str]:
    """
    Собирает подряд идущие сверху комментарии (// ... или /* ... */),
    непосредственно прилегающие к node, без пустых строк между ними.
    Текущая реализация оставлена простой: ищем среди комментов предков.
    """
    lines = []
    prev_end_row = node.start_point[0] - 1  # строка перед узлом
    parent = node.parent
    if parent is None:
        return None

    # Пробегаем только по комментариям под тем же родителем (не по всему файлу)
    for com in _all_descendants_of_type(parent, "comment"):
        if com.end_point[0] == prev_end_row:
            text = _text(src, com)
            lines.insert(0, text)  # вставляем в начало, чтобы сохранить порядок сверху-вниз
            # уменьшаем prev_end_row на число строк комментария
            # (для /* ... */ многострочных блоков)
            prev_end_row -= (text.count("\n") or 0)

    return "\n".join(lines) if lines else None


# ──────────────────────────────────────────────────────────────────────────────
# AST helpers — без Query
# ──────────────────────────────────────────────────────────────────────────────

def _children(node):
    # Простой генератор по дочерним
    for ch in node.children:
        yield ch


def _first_child_of_type(node, t: str):
    for ch in node.children:
        if ch.type == t:
            return ch
    return None


def _all_descendants_of_type(node, t: str):
    stack = [node]
    while stack:
        n = stack.pop()
        if n.type == t:
            yield n
        # extend без создания лишних списков
        if n.children:
            stack.extend(n.children)

In [76]:
# ──────────────────────────────────────────────────────────────────────────────
# Парсер типов и вложенных структур (+ хелперы для proto)
# ──────────────────────────────────────────────────────────────────────────────

def _extract_field_tag(src: bytes, decl_node) -> Optional[str]:
    """
    Возвращает строку тега поля (без кавычек) если в декларации поля есть завершающий string literal.
    В tree-sitter-go теги представлены как raw|interpreted string literal в конце декларации.
    """
    tag: Optional[str] = None
    for ch in _children(decl_node):
        if ch.type in ("interpreted_string_literal", "raw_string_literal"):
            tag = _strip_quotes(_text(src, ch))
    return tag


def _first_type_node_in_field_decl(decl_node):
    """
    Возвращает первый узел типа в field_declaration:
    (struct_type | pointer_type | slice_type | qualified_type | array_type | map_type | type_identifier | ...)
    """
    for ch in _children(decl_node):
        if ch.type in (
            "struct_type", "pointer_type", "slice_type", "qualified_type",
            "array_type", "map_type", "channel_type", "generic_type",
            "type_identifier", "union_type", "function_type",
        ):
            return ch
    return None


# ──────────────────────────────────────────────────────────────────────────────
# Хелперы распаковки типов до базового идентификатора (для proto matching)
# ──────────────────────────────────────────────────────────────────────────────

def unwrap_type_ident(src: bytes, type_node) -> Tuple[Optional[str], Optional[str]]:
    """
    Разворачивает тип до базового идентификатора и pkg-алиаса.
    Возвращает кортеж (pkg_alias, type_name), где pkg_alias может быть None.

    Поддержка:
      * pointer_type: *T
      * slice_type/array_type: []T / [N]T
      * channel_type: chan T
      * qualified_type: pkg.T
      * type_identifier: T
      * generic_type: Foo[Bar] -> берём Foo как идентификатор
    """
    if type_node is None:
        return None, None

    n = type_node
    # снимаем слои-обёртки
    while n.type in ("pointer_type", "slice_type", "array_type", "channel_type", "parenthesized_type"):
        # у всех этих узлов тип лежит в единственном child-е типа
        ch = _first_type_node_in_field_decl(n) or (n.children[0] if n.children else None)
        if ch is None:
            return None, None
        n = ch

    if n.type == "qualified_type":
        # qualified_type обычно: package_identifier "." type_identifier
        pkg_ident = _first_child_of_type(n, "package_identifier")
        typ_ident = _first_child_of_type(n, "type_identifier")
        pkg_alias = _text(src, pkg_ident) if pkg_ident else None
        type_name = _text(src, typ_ident) if typ_ident else None
        return pkg_alias, type_name

    if n.type == "generic_type":
        # generic_type: идентификатор с параметрами (Go 1.18+). Берём базовое имя типа.
        base = _first_child_of_type(n, "type_identifier") or _first_child_of_type(n, "qualified_type")
        if base is None:
            return None, None
        return unwrap_type_ident(src, base)

    if n.type == "type_identifier":
        return None, _text(src, n)

    # В остальных случаях (map_type, function_type, struct_type, union_type)
    # это не proto message имя.
    return None, None


def detect_proto_ref(
    pkg_alias: Optional[str],
    type_name: Optional[str],
    proto_index: Optional[Dict[str, object]] = None,
    known_proto_aliases: Optional[Set[str]] = None,
):
    """
    Пытается сопоставить (pkg_alias, type_name) с proto message через индекс.
    Ожидаемый формат proto_index ключей:
      - "Alias.MessageName" (например, "pb.GetUserRequest")
      - "MessageName"       (на случай уникальных имён)
    Возвращает объект из индекса (например, ProtoMessage) или None.
    """
    if not proto_index or not type_name:
        return None

    # приоритет: alias-qualified
    if pkg_alias:
        key = f"{pkg_alias}.{type_name}"
        if key in proto_index:
            return proto_index[key]

    # если алиас не задан, но это точно pb-алиас (в сигнатурах методов часто пишут "*pb.X")
    if known_proto_aliases and pkg_alias in known_proto_aliases:
        key = f"{pkg_alias}.{type_name}"
        if key in proto_index:
            return proto_index[key]

    # fallback — по голому имени (если уникально)
    return proto_index.get(type_name)


# ──────────────────────────────────────────────────────────────────────────────
# Разбор полей структуры
# ──────────────────────────────────────────────────────────────────────────────

def _parse_struct_fields(
    src: bytes,
    field_decl_list_node,
    embedded: bool = False,
    *,
    proto_index: Optional[Dict[str, object]] = None,
    known_proto_aliases: Optional[Set[str]] = None,
) -> List[GoField]:
    """
    Разбирает field_declaration_list внутри struct_type -> список GoField.
    Поддерживает:
      - несколько имён на один тип: A, B int
      - embedded поля: *Other / Other / pkg.Type
      - inline struct: X struct { ... }
      - теги у полей
    Опционально: proto_index/known_proto_aliases для последующего сопоставления
    (детект не записывается в GoField; если нужно — добавим поле в модель).
    """
    out: List[GoField] = []

    for decl in _all_descendants_of_type(field_decl_list_node, "field_declaration"):
        # Имена (могут быть отсутствующими для embedded)
        names = [_text(src, n) for n in _all_descendants_of_type(decl, "field_identifier")]
        embedded_node = _first_child_of_type(decl, "embedded_field")
        tag = _extract_field_tag(src, decl)
        type_node = _first_type_node_in_field_decl(decl)

        # Тип как текст (важно сохранить исходный вид, включая "struct {...}")
        type_text = _text(src, type_node).strip() if type_node else ""

        # Если это inline struct, рекурсивно распарсим вложенные поля
        subfields: Optional[List[GoField]] = None
        if type_node and type_node.type == "struct_type":
            fld_list = _first_child_of_type(type_node, "field_declaration_list")
            if fld_list:
                subfields = _parse_struct_fields(
                    src, fld_list, True,
                    proto_index=proto_index,
                    known_proto_aliases=known_proto_aliases,
                )

        # ── Proto detection (опционально, без побочных эффектов) ─────────────
        pkg_alias, base_type = unwrap_type_ident(src, type_node) if type_node else (None, None)
        _proto_msg = detect_proto_ref(pkg_alias, base_type, proto_index, known_proto_aliases)
        # Примечание: мы не сохраняем _proto_msg в GoField — следуем текущей модели.
        # Если потребуется — можно расширить GoField отдельным полем.

        if embedded_node and not names:
            # embedded поле — используем тип как имя
            emb_text = _text(src, embedded_node).strip()
            out.append(GoField(
                name=emb_text,
                type=type_text or emb_text,
                tag=tag,
                embedded=embedded,
                fields=subfields
            ))
            continue

        # Обычные именованные поля (возможны несколько имён под один тип)
        if names:
            for nm in names:
                out.append(GoField(
                    name=nm,
                    type=type_text,
                    tag=tag,
                    embedded=embedded,
                    fields=subfields
                ))
        else:
            # Теоретически: анонимное поле без embedded (редко), но поддержим
            out.append(GoField(
                name=type_text or "<anonymous>",
                type=type_text,
                tag=tag,
                embedded=embedded,
                fields=subfields
            ))

    return out

In [77]:
# ──────────────────────────────────────────────────────────────────────────────
# Парсер параметров в функциях (точечный тюнинг без смены логики)
# ──────────────────────────────────────────────────────────────────────────────

# Какие узлы считаем "шумом" при поиске типа
_SKIP_IN_TYPE = {"identifier", "comment", ",", "ellipsis"}  # ellipsis = '...'

def _node_sexpr(src: bytes, node) -> str:
    # Универсальный способ получить «текст типа» для узла
    return _text(src, node).strip()


def _first_type_child(node):
    """
    Возвращает первый дочерний узел, который выглядит как тип (а не имя/комментарий/запятая/ellipsis).
    Работает и для parameter_declaration, и для variadic_parameter_declaration.
    """
    for ch in node.children:
        if ch.type not in _SKIP_IN_TYPE:
            return ch
    return None


def _param_type_text(src: bytes, decl_node) -> Optional[str]:
    """
    В parameter_declaration тип — первый осмысленный дочерний узел (не имя/коммент/запятая).
    """
    tnode = _first_type_child(decl_node)
    if tnode is None:
        return None
    tt = _text(src, tnode).strip()
    return tt or None


def _parse_parameter_list(src: bytes, plist_node) -> List[GoParam]:
    """
    Разбирает (parameter_list) → [GoParam,...]
    Поддерживает:
      - parameter_declaration с несколькими идентификаторами, разделяющими один тип
      - variadic_parameter_declaration ( ...T / name ...T )
      - анонимные параметры (только тип, без имени)
    """
    params: List[GoParam] = []

    for ch in _children(plist_node):
        t = ch.type
        if t == "parameter_declaration":
            # Имена строго из прямых детей-типа identifier (не из qualified_type)
            direct_names = [ _text(src, n) for n in ch.children if n.type == "identifier" ]
            ptype = _param_type_text(src, ch) or ""
            if direct_names:
                for nm in direct_names:
                    params.append(GoParam(name=nm, type=ptype, variadic=False))
            else:
                # анонимный параметр
                params.append(GoParam(name=None, type=ptype, variadic=False))

        elif t == "variadic_parameter_declaration":
            # Вариадик: возможны формы "name ...T" и "...T"
            # Тип берём как первый осмысленный дочерний узел (после '...' и, возможно, имени)
            direct_name = None
            for c2 in ch.children:
                if c2.type == "identifier":
                    direct_name = _text(src, c2)
                    break
            tnode = _first_type_child(ch)
            ptype = _text(src, tnode).strip() if tnode else ""
            params.append(GoParam(name=direct_name, type=ptype, variadic=True))

        else:
            # пропускаем запятые/скобки и т.п.
            continue

    return params


def _parse_results(src: bytes, fn_or_sig_node) -> List[str]:
    """
    Ищем возвращаемую часть после списка параметров.
    В Go это либо единичный тип, либо (parameter_list) с именованными/безымянными результатами.
    Стратегия:
      1) Найти первый (parameter_list) — это входные параметры.
      2) После него:
         - если следующий узел — (parameter_list): это результаты (берём типы с параметров),
         - иначе ищем первый осмысленный узел-типа (до тела функции/блока).
    """
    results: List[str] = []
    children = fn_or_sig_node.children  # без лишнего list()

    # Найти индекс первого parameter_list (вход)
    first_pl_idx = -1
    for i, n in enumerate(children):
        if n.type == "parameter_list":
            first_pl_idx = i
            break
    if first_pl_idx == -1:
        return results  # на всякий случай (в обычных func сигнатура всегда есть)

    # Хвост после входного списка
    tail = children[first_pl_idx + 1:]
    if not tail:
        return results

    # Случай 1: следующий узел — тоже parameter_list → это выходные параметры
    if tail[0].type == "parameter_list":
        out_pl = tail[0]
        for ch in _children(out_pl):
            if ch.type in ("parameter_declaration", "variadic_parameter_declaration"):
                t = _param_type_text(src, ch)
                if t:
                    results.append(t)
        return results

    # Случай 2: одиночный тип → берём первый осмысленный узел до тела функции
    for n in tail:
        # Останавливаемся перед блоками/телом/where_clause (generics), если встретятся
        if n.type in ("block",):
            break
        if n.type not in ("comment", ","):
            results.append(_node_sexpr(src, n))
            break

    return results

In [78]:
# ──────────────────────────────────────────────────────────────────────────────
# Парсинг одного файла — только навигация по типам узлов (микро-оптимизация)
# ──────────────────────────────────────────────────────────────────────────────

# константы для проверок типов узлов (ускоряют membership)
_STR_LITS = ("interpreted_string_literal", "raw_string_literal")

def parse_go_file_structures(path: Path, parser: Parser) -> GoFileMeta:
    src = path.read_bytes()
    tree = parser.parse(src)
    root = tree.root_node

    # локальные ссылки (чуть быстрее, меньше глобальных lookup)
    _t = _text
    _fc = _first_child_of_type
    _ad = _all_descendants_of_type

    # ---- package ----
    package: Optional[str] = None
    pkg_clause = _fc(root, "package_clause")
    if pkg_clause:
        ident = _fc(pkg_clause, "package_identifier") or _fc(pkg_clause, "identifier")
        if ident:
            package = _t(src, ident)

    # ---- imports ----
    imports: List[GoImport] = []
    for imp_decl in _ad(root, "import_declaration"):
        for spec in _ad(imp_decl, "import_spec"):
            alias: Optional[str] = None
            path_lit: Optional[str] = None
            # порядок важен: alias (identifier|_) может стоять перед строковым литералом
            for ch in spec.children:
                ct = ch.type
                if ct == "identifier" and alias is None:
                    alias = _t(src, ch)
                elif ct in _STR_LITS and path_lit is None:
                    # убираем кавычки/бэктики
                    path_lit = _string_literal_text(src, ch)
                # небольшая оптимизация: выходим, когда уже нашли оба
                if alias is not None and path_lit is not None:
                    break
            if path_lit:
                imports.append(GoImport(path=path_lit, alias=alias))

    # ---- types ----
    types: List[GoType] = []
    for type_decl in _ad(root, "type_declaration"):
        for type_spec in _ad(type_decl, "type_spec"):
            tname_node = _fc(type_spec, "type_identifier")
            if not tname_node:
                continue
            tname = _t(src, tname_node)
            if not tname:
                continue

            line = type_spec.start_point[0] + 1
            struct_node = _fc(type_spec, "struct_type")
            if struct_node is not None:
                fld_list = _fc(struct_node, "field_declaration_list")
                fields = _parse_struct_fields(src, fld_list) if fld_list else []
                # сохраняем ровно то же, что было: list GoField (не меняем модель)
                types.append(GoType(name=tname, kind="struct", fields=fields or None, line=line))
                continue

            iface_node = _fc(type_spec, "interface_type")
            if iface_node is not None:
                methods: List[str] = []
                # method_spec может лежать на разной глубине — ищем потомков
                for ms in _ad(iface_node, "method_spec"):
                    methods.append(_t(src, ms).strip())
                types.append(GoType(name=tname, kind="interface", methods=methods or None, line=line))
                continue

            # alias / other: берём RHS как сырой текст (первый непустой кусок после имени)
            rhs: Optional[str] = None
            for ch in type_spec.children:
                if ch is tname_node:
                    continue
                txt = _t(src, ch).strip()
                if txt:
                    rhs = txt
                    break
            if rhs:
                types.append(GoType(name=tname, kind="alias", methods=[rhs], line=line))
            else:
                types.append(GoType(name=tname, kind="other", line=line))

    # ---- funcs & methods ----
    funcs: List[GoFunc] = []

    # Свободные функции
    for fn in _ad(root, "function_declaration"):
        fname_node = _fc(fn, "identifier")
        if not fname_node:
            continue
        fname = _t(src, fname_node)
        if not fname:
            continue

        in_pl = _fc(fn, "parameter_list")
        params = _parse_parameter_list(src, in_pl) if in_pl else []
        results = _parse_results(src, fn)
        doc = _collect_leading_comments(src, fn)
        line = fn.start_point[0] + 1

        funcs.append(GoFunc(
            name=fname,
            receiver=None,
            exported=(fname[0].isupper() if fname else False),
            params=params,
            results=results,
            doc=doc,
            line=line,
        ))

    # Методы (с ресивером)
    for md in _ad(root, "method_declaration"):
        # имя метода может быть field_identifier (обычно) или identifier (реже)
        mname_node = _fc(md, "field_identifier") or _fc(md, "identifier")
        if not mname_node:
            continue
        mname = _t(src, mname_node)
        if not mname:
            continue

        # ресивер — первый parameter_list
        recv: Optional[str] = None
        # вместо создания списка plists — найдём первый/второй на лету
        first_pl = None
        second_pl = None
        for ch in md.children:
            if ch.type == "parameter_list":
                if first_pl is None:
                    first_pl = ch
                elif second_pl is None:
                    second_pl = ch
                    break  # больше не нужно

        if first_pl is not None:
            # берём первый «осмысленный» тип из первой parameter_declaration
            for pd in _ad(first_pl, "parameter_declaration"):
                # первый ребёнок, который не identifier/comment — это и есть тип ресивера
                for ch in pd.children:
                    if ch.type not in ("identifier", "comment"):
                        rt = _t(src, ch).strip()
                        if rt:
                            recv = rt
                            break
                if recv:
                    break

        # параметры метода — второй parameter_list (если есть)
        params: List[GoParam] = _parse_parameter_list(src, second_pl) if second_pl is not None else []

        results = _parse_results(src, md)
        doc = _collect_leading_comments(src, md)
        line = md.start_point[0] + 1

        funcs.append(GoFunc(
            name=mname,
            receiver=recv,
            exported=(mname[0].isupper() if mname else False),
            params=params,
            results=results,
            doc=doc,
            line=line,
        ))

    return GoFileMeta(
        path=str(path),
        package=package,
        imports=imports,
        types=types,
        funcs=funcs,
    )

In [79]:
# ──────────────────────────────────────────────────────────────────────────────
# Публичный контракт (микро-оптимизация без смены логики)
# ──────────────────────────────────────────────────────────────────────────────
from typing import Dict, Iterable, Union
from pathlib import Path

def make_project_tree(PROJECT_PATH: Union[str, Path], IGNORE_DIRS: Iterable[str]) -> Dict:
    """
    Возвращает файловое дерево (dict) только с *.go, go.mod, go.sum.
    """
    root = Path(PROJECT_PATH).resolve()
    tree: Dict = {"name": root.name, "type": "dir", "children": {}}

    children = tree["children"]  # локальная ссылка

    def insert(path: Path):
        rel = path.relative_to(root)
        node = children
        parts = rel.parts
        for i, part in enumerate(parts):
            is_last = i == len(parts) - 1
            nd = node.get(part)
            if nd is None:
                nd = node[part] = {
                    "name": part,
                    "type": "file" if is_last else "dir",
                    "children": None if is_last else {},
                }
            if not is_last:
                node = nd["children"]

    # детерминированный порядок
    files = sorted(iter_go_files(root, IGNORE_DIRS), key=lambda p: (p.parent.as_posix(), p.name))
    for f in files:
        insert(f)

    return tree


def make_structures(PROJECT_PATH: Union[str, Path], IGNORE_DIRS: Iterable[str]) -> Dict[str, GoFileMeta]:
    """
    Возвращает {relative_path: GoFileMeta} по всем *.go с разбором пакетов/импортов/типов/функций.
    """
    root = Path(PROJECT_PATH).resolve()

    # используем ваш фабричный метод; если в ноутбуке уже есть make_go_parser() — применим его
    try:
        parser = make_go_parser()  # из предыдущей ячейки
    except NameError:
        parser = _make_parser()    # ваш исходный вариант

    out: Dict[str, GoFileMeta] = {}
    # только .go файлы; детерминированный порядок
    files = sorted(iter_go_files(root, IGNORE_DIRS), key=lambda p: (p.parent.as_posix(), p.name))
    rel = Path.relative_to  # локальная ссылка на метод для микро-экономии лукапов

    for f in files:
        if f.suffix != ".go":
            continue
        meta = parse_go_file_structures(f, parser)
        out[str(rel(f, root))] = meta

    return out

In [82]:
# ──────────────────────────────────────────────────────────────────────────────
# Утилиты вывода (микро-оптимизация без изменения API/форматов)
# ──────────────────────────────────────────────────────────────────────────────

from dataclasses import asdict
import json
from typing import Dict, List

def to_jsonable_structures(structs: Dict[str, GoFileMeta]) -> Dict[str, Dict]:
    # Локальные ссылки уменьшают оверхед на глобальные лукапы в больших словарях
    _asdict = asdict
    return {k: _asdict(v) for k, v in structs.items()}


def dump_structures_json(structs: Dict[str, GoFileMeta]) -> str:
    # Те же параметры, но минимизируем лишние лукапы
    return json.dumps(
        to_jsonable_structures(structs),
        ensure_ascii=False,
        indent=2,
    )


def tree_to_pretty_lines(tree: Dict, indent: str = "") -> List[str]:
    """
    Формирует список строк вида:
      root/
        dir1/
          file.go
        go.mod
    Предполагается, что каталог имеет "type": "dir", файл — "type": "file".
    """
    lines: List[str] = []
    _children = tree.get("children") or {}
    name = tree.get("name", "")
    node_type = tree.get("type", "dir")

    # Заголовок узла
    if node_type == "dir":
        lines.append(f"{indent}{name}/")
    else:
        # На случай если корень передали как file
        lines.append(f"{indent}{name}")

    # Дети (детерминированный порядок)
    for child_name in sorted(_children.keys()):
        node = _children[child_name]
        ntype = node.get("type", "file")
        if ntype == "dir":
            # Хвост рекурсивно
            lines.extend(tree_to_pretty_lines(node, indent + "  "))
        else:
            lines.append(f"{indent}  {child_name}")

    return lines


# proto анализ


In [83]:
# ──────────────────────────────────────────────────────────────────────────────
# Привязка внешних import-путей .proto к локальным каталогам
# ──────────────────────────────────────────────────────────────────────────────
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import re


# Пример: внешний импорт → локальный корень (можно несколько)
# Ключ — префикс import-пути из .go ("gitlab.com/kuber/proto"), значение — локальный путь
PROTO_PATH_MAP: Dict[str, Path] = {
    # "gitlab.com/kuber/proto": Path("/home/jovyan/work/third_party/kuber/proto"),
}

# Регекс для Go-типа alias.Type или *alias.Type / []alias.Type (оставляем как было)
_GO_QTYPE = re.compile(r'^\s*(?:\*|\[\])*?\s*(?:(?P<alias>[A-Za-z_]\w*)\.)?(?P<type>[A-Za-z_]\w+)\s*$')

def _split_go_qual_type(t: str) -> Optional[Tuple[Optional[str], str]]:
    m = _GO_QTYPE.match(t)
    if not m:
        return None
    return (m.group('alias'), m.group('type'))


def _resolve_import_to_local(import_path: str,
                             path_map: Dict[str, Path],
                             include_paths: List[Path]) -> Optional[Path]:
    """
    Стратегия поиска локального файла .proto для import-пути вида
      gitlab.com/kuber/proto/path/to/protfile
    1) PROTO_PATH_MAP: ищем самый длинный совпавший префикс и подставляем локальный корень
       → <mapped_root>/<suffix>.proto
    2) include_paths: прямое соответствие → <inc>/<import_path>.proto
    3) fallback: суффиксный поиск по include_paths (на случай рассинхрона директорий)
    """
    norm = import_path.strip("/").replace("\\", "/")
    # 1) карта префиксов (самое длинное совпадение)
    best_prefix = ""
    best_root: Optional[Path] = None
    for prefix, root in path_map.items():
        pfx = prefix.strip("/").replace("\\", "/")
        if norm == pfx or norm.startswith(pfx + "/"):
            if len(pfx) > len(best_prefix):
                best_prefix, best_root = pfx, root
    if best_root is not None:
        suffix = norm[len(best_prefix):].lstrip("/")
        cand = best_root.joinpath(suffix).with_suffix(".proto")
        if cand.is_file():
            return cand

    # 2) прямой путь через include_paths
    for base in include_paths:
        cand = base.joinpath(norm).with_suffix(".proto")
        if cand.is_file():
            return cand

    # 3) суффиксный fallback-поиск (дороже: пробегаем дерево однократно)
    suffix = norm.split("/")[-1] + ".proto"  # имя файла
    for base in include_paths:
        for p in base.rglob(suffix):
            # проверим, что хвост пути совпадает по сегментам (чтобы не схватить одноимённый файл)
            # сравниваем по суффиксу из исходного import_path
            tail = norm.split("/")
            parts = p.as_posix().split("/")
            # грубая, но быстрая проверка суффикса
            if len(parts) >= len(tail) and parts[-len(tail):] == tail:
                return p
    return None


def _build_alias_to_proto_files(structs_map: Dict[str, dict],
                                include_paths: List[Path],
                                path_map: Dict[str, Path]) -> Dict[str, ProtoFileMeta]:
    """
    По всем Go-файлам берём импорты с алиасом (alias "path"), находим соответствующий .proto
    (через PROTO_PATH_MAP + include_paths) и парсим его.
    Возвращает индекс alias -> ProtoFileMeta.
    """
    alias_to_proto: Dict[str, ProtoFileMeta] = {}
    for rel, meta in structs_map.items():
        for imp in (meta.get("imports") or []):
            alias = imp.get("alias")
            path = imp.get("path")
            if not alias or not path or alias in alias_to_proto:
                continue
            p = _resolve_import_to_local(path, path_map, include_paths)
            if not p:
                continue
            pf = parse_proto_file(p)
            alias_to_proto[alias] = pf
    return alias_to_proto


def _build_message_index_by_alias(alias_to_proto: Dict[str, ProtoFileMeta]) -> Dict[str, ProtoMessage]:
    """
    Делает ключи вида "alias.Message" → ProtoMessage
    Плюс безалиасный ключ "Message" на случай уникальных имён.
    """
    idx: Dict[str, ProtoMessage] = {}
    for alias, pf in alias_to_proto.items():
        for m in pf.messages:
            idx[f"{alias}.{m.name}"] = m
            if m.name not in idx:
                idx[m.name] = m
    return idx


# маппинг скаляров (осталось как было)
_PROTO_SCALARS_TO_GO = {
    "double": "float64", "float": "float32",
    "int32": "int32", "int64": "int64", "uint32": "uint32", "uint64": "uint64",
    "sint32": "int32", "sint64": "int64", "fixed32": "uint32", "fixed64": "uint64",
    "sfixed32": "int32", "sfixed64": "int64",
    "bool": "bool", "string": "string", "bytes": "[]byte",
}

def _proto_field_to_gofield(pf: ProtoField) -> GoField:
    base = _PROTO_SCALARS_TO_GO.get(pf.type, pf.type)
    go_type = f"[]{base}" if (pf.label == "repeated") else base
    return GoField(name=pf.name, type=go_type, tag=None, embedded=False, fields=None)


def expand_proto_fields_in_structs(structs_map: Dict[str, dict],
                                   include_paths: List[Path],
                                   path_map: Dict[str, Path]) -> int:
    """
    Модифицирует structs_map IN-PLACE:
      Поля Go-структур типа alias.Message разворачиваются в список полей message из .proto.
    Резолв импортов выполняется через PROTO_PATH_MAP и include_paths.
    """
    alias_to_proto = _build_alias_to_proto_files(structs_map, include_paths, path_map)
    msg_index = _build_message_index_by_alias(alias_to_proto)
    expanded = 0

    for _, meta in structs_map.items():
        for t in meta.get("types") or []:
            if t.get("kind") != "struct":
                continue
            for fld in t.get("fields") or []:
                if fld.get("fields"):  # уже инлайн-struct
                    continue
                tp = fld.get("type") or ""
                sk = _split_go_qual_type(tp)
                if not sk:
                    continue
                alias, typ = sk
                key = f"{alias}.{typ}" if alias else typ
                pm = msg_index.get(key)
                if not pm:
                    continue
                fld["fields"] = [_proto_field_to_gofield(pf).__dict__ for pf in pm.fields]
                expanded += 1
    return expanded

# PlantUML диаграма

In [84]:
# ──────────────────────────────────────────────────────────────────────────────
# PlantUML: struct/interface + component(функции), allowmixing, no *_test.go,
#           без init() и без пустых function-компонентов
# ──────────────────────────────────────────────────────────────────────────────
import json
from typing import Any, Dict, List, Tuple, Set

def plantuml_components_allowmixing_filtered(data: Dict[str, Any] | str) -> str:
    if _is_json_str(data):
        data = json.loads(data)
    structs_map: Dict[str, Any] = data  # type: ignore

    # 0) исключаем тестовые файлы
    files = {k: v for k, v in structs_map.items() if not k.endswith("_test.go")}

    # 1) индексация типов
    struct_names, alias_rhs, struct_items, name_counts = _index_structs_and_aliases(files)

    # индексы
    class_alias_of_base: Dict[str, str] = {}
    struct_alias_display: Dict[str, str] = {}
    iface_alias_display: Dict[str, str] = {}
    func_alias_display: Dict[str, str] = {}

    struct_by_file: Dict[str, List[str]] = {}
    iface_by_file: Dict[str, List[str]] = {}
    func_by_file: Dict[str, List[str]] = {}

    struct_type_by_alias: Dict[str, Any] = {}
    func_meta_by_alias: Dict[str, Any] = {}

    # 2) struct
    for rel_path, pkg, t in struct_items:
        name = _get(t, "name")
        disp = name if name_counts.get(name, 0) == 1 else (f"{pkg}.{name}" if pkg else name)
        alias = _sanitize_ident(f"struct_{disp}")
        struct_alias_display[alias] = disp
        struct_by_file.setdefault(rel_path, []).append(alias)
        struct_type_by_alias[alias] = t
        class_alias_of_base.setdefault(name, alias)

    # 3) interface
    for rel_path, meta in files.items():
        pkg = _get(meta, "package", "") or ""
        for t in _iter(meta, "types"):
            if _get(t, "kind") != "interface":
                continue
            iname = _get(t, "name")
            if not iname:
                continue
            disp = iname if name_counts.get(iname, 0) == 1 else (f"{pkg}.{iname}" if pkg else iname)
            alias = _sanitize_ident(f"iface_{disp}")
            iface_alias_display[alias] = disp
            iface_by_file.setdefault(rel_path, []).append(alias)

    # 4) functions/methods → component (без init)
    for rel_path, meta in files.items():
        for fn in _iter(meta, "funcs"):
            fname = _get(fn, "name", "")
            if fname == "init":
                continue  # ⟵ скрываем init()
            recv  = _get(fn, "receiver", None)
            in_sig  = _fmt_params_for_sig(_iter(fn, "params"))
            out_sig = _fmt_results_for_sig(_get(fn, "results", []) or [])
            if recv:
                recv_short = _short_typename(recv)
                disp = f"({recv_short}) {fname}({in_sig}){out_sig}"
            else:
                disp = f"{fname}({in_sig}){out_sig}"
            alias = _sanitize_ident(f"fn_{rel_path}__{disp}")
            func_alias_display[alias] = disp.replace('"', "'")
            func_by_file.setdefault(rel_path, []).append(alias)
            func_meta_by_alias[alias] = {
                "receiver": recv,
                "params": _iter(fn, "params"),
                "results": _get(fn, "results", []) or [],
            }

    # 5) связи
    edges: List[str] = []

    # struct поля: вложенность/ссылки
    for s_alias, t in struct_type_by_alias.items():
        for fld in _iter(t, "fields"):
            tgt_name = _resolve_to_struct_name(_get(fld, "type", ""), struct_names, alias_rhs)
            if not tgt_name:
                continue
            tgt_alias = class_alias_of_base.get(tgt_name)
            if not tgt_alias or tgt_alias == s_alias:
                continue
            if bool(_get(fld, "embedded", False)):
                edges.append(f"  {s_alias} *-- {tgt_alias}")
            else:
                edges.append(f"  {s_alias} o-- {tgt_alias}")

    # функции ↔ структуры: in/out/receiver
    # одновременно считаем «ненулевые» функции (имеют хотя бы одно ребро)
    non_empty_funcs: Set[str] = set()

    for f_alias, fn in func_meta_by_alias.items():
        # in
        for p in fn["params"]:
            tgt_name = _resolve_to_struct_name(_get(p, "type", ""), struct_names, alias_rhs)
            if tgt_name:
                tgt_alias = class_alias_of_base.get(tgt_name)
                if tgt_alias:
                    edges.append(f"  {f_alias} ..> {tgt_alias} : in")
                    non_empty_funcs.add(f_alias)
        # out
        for r in fn["results"]:
            tgt_name = _resolve_to_struct_name(r, struct_names, alias_rhs)
            if tgt_name:
                tgt_alias = class_alias_of_base.get(tgt_name)
                if tgt_alias:
                    edges.append(f"  {f_alias} ..> {tgt_alias} : out")
                    non_empty_funcs.add(f_alias)
        # receiver
        recv = fn.get("receiver")
        if recv:
            base = _short_typename(recv)
            tgt_alias = class_alias_of_base.get(base)
            if tgt_alias:
                edges.append(f"  {f_alias} -- {tgt_alias} : receiver")
                non_empty_funcs.add(f_alias)

    # 6) генерация PlantUML
    lines: List[str] = [
        "@startuml",
        "allowmixing",
        "hide empty members",
    ]

    # выводим только пакеты, в которых после фильтра остались элементы
    for rel_path in sorted(files.keys()):
        structs_in = sorted(struct_by_file.get(rel_path, []))
        ifaces_in  = sorted(iface_by_file.get(rel_path, []))
        # функции оставляем только «непустые»
        funcs_all  = sorted(func_by_file.get(rel_path, []))
        funcs_in   = [a for a in funcs_all if a in non_empty_funcs]

        if not (structs_in or ifaces_in or funcs_in):
            continue  # ⟵ пустой пакет пропускаем

        pkg_alias = _sanitize_ident(f"file_{rel_path}")
        lines.append(f'package "{rel_path}" as {pkg_alias} {{')

        for a in structs_in:
            disp = struct_alias_display[a]
            lines.append(f'  struct "{disp}" as {a}')

        for a in ifaces_in:
            disp = iface_alias_display[a]
            lines.append(f'  interface "{disp}" as {a}')

        for a in funcs_in:
            disp = func_alias_display[a]
            lines.append(f'  component "{disp}" as {a}')

        lines.append("}")

    # оставляем только те рёбра, которые ссылаются на реально выведенные узлы
    rendered_nodes: Set[str] = set()
    for rel_path in sorted(files.keys()):
        rendered_nodes.update(struct_by_file.get(rel_path, []))
        rendered_nodes.update(iface_by_file.get(rel_path, []))
        rendered_nodes.update([a for a in func_by_file.get(rel_path, []) if a in non_empty_funcs])

    for e in edges:
        # грубая фильтрация: рёбра вида "  A op B ..." — достанем A и B
        parts = e.strip().split()
        if len(parts) >= 3:
            a, b = parts[0], parts[2]
            if a in rendered_nodes and b in rendered_nodes:
                lines.append(e)

    lines += [
        "legend left",
        "  struct     — Go struct",
        "  interface  — Go interface",
        "  component  — Go function/method (без init, без пустых)",
        "  struct *-- struct  — embedded",
        "  struct o-- struct  — field ref",
        "  component ..> struct : in/out",
        "  component -- struct : receiver",
        "endlegend",
        "@enduml",
    ]
    return "\n".join(lines)

# Полный пайплайн: Go → JSON → расширение через .proto → PlantUML

In [87]:
# ──────────────────────────────────────────────────────────────────────────────
# Полный пайплайн: Go → JSON → расширение через .proto → PlantUML
# ──────────────────────────────────────────────────────────────────────────────
import json, time
from pathlib import Path

# === 0) Конфигурация путей ===
PROJECT_PATH = "/home/jovyan/work/tree_docs/example/k8sgpt/"
OUT_DIR      = "/home/jovyan/work/tree_docs/example/"
RESULT_JSON  = str(Path(OUT_DIR) / "result.json")
RESULT_EXP   = str(Path(OUT_DIR) / "result_expanded.json")
PUML_PATH    = str(Path(OUT_DIR) / "component_allowmixing.puml")

# Пути поиска .proto
PROTO_PATH_MAP = {
    "gitlab.com/kuber/proto": Path("/home/jovyan/work/third_party/kuber/proto")
}

PROTO_INCLUDE_PATHS = [Path(PROJECT_PATH), Path("/home/jovyan/work/third_party")]

IGNORE_DIRS = {'.git', 'vendor', 'node_modules', 'bin', 'dist', 'out', 'build', '.idea', '.vscode'}

t0 = time.time()

# === 1) Парсинг Go-структур ===
structs = make_structures(PROJECT_PATH, IGNORE_DIRS)
t1 = time.time()

# === 2) Сохранение исходного результата ===
json_result = dump_structures_json(structs)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
with open(RESULT_JSON, "w", encoding="utf-8") as f:
    json.dump(json.loads(json_result), f, ensure_ascii=False, indent=4)

# === 3) Пост-обработка: разворачиваем поля из .proto ===
with open(RESULT_JSON, "r", encoding="utf-8") as f:
    go_structs_dict = json.load(f)

expanded = expand_proto_fields_in_structs(go_structs_dict, PROTO_INCLUDE_PATHS, PROTO_PATH_MAP)
print(f"Развёрнуто полей из .proto: {expanded}")

with open(RESULT_EXP, "w", encoding="utf-8") as f:
    json.dump(go_structs_dict, f, ensure_ascii=False, indent=2)

t2 = time.time()

# === 4) Генерация PlantUML ===
puml = plantuml_components_allowmixing_filtered(go_structs_dict)
with open(PUML_PATH, "w", encoding="utf-8") as f:
    f.write(puml)

t3 = time.time()

# === 5) Отчёт ===
files_total   = len(structs)
types_total   = sum(len(m.types or []) for m in structs.values())
funcs_total   = sum(len(m.funcs or []) for m in structs.values())
imports_total = sum(len(m.imports or []) for m in structs.values())

print("— Пайплайн завершён —")
print(f"Go-файлов обработано : {files_total}")
print(f"Импортов найдено     : {imports_total}")
print(f"Типов (struct/iface) : {types_total}")
print(f"Функций/методов      : {funcs_total}")
print(f"Расширено proto-полей: {expanded_count}")
print()
print(f"Сохранён JSON (Go)   : {RESULT_JSON}")
print(f"Сохранён JSON+proto  : {RESULT_EXP}")
print(f"Сохранена диаграмма  : {PUML_PATH}")
print()
print(f"Время: парсинг {t1-t0:.3f}с | proto {t2-t1:.3f}с | диаграмма {t3-t2:.3f}с | всего {t3-t0:.3f}с")

print("\n--- Начало PlantUML ---")
print("\n".join(puml.splitlines()[:3]))
print("--- ... ---")

Развёрнуто полей из .proto: 0
— Пайплайн завершён —
Go-файлов обработано : 169
Импортов найдено     : 1116
Типов (struct/iface) : 193
Функций/методов      : 563
Расширено proto-полей: 0

Сохранён JSON (Go)   : /home/jovyan/work/tree_docs/example/result.json
Сохранён JSON+proto  : /home/jovyan/work/tree_docs/example/result_expanded.json
Сохранена диаграмма  : /home/jovyan/work/tree_docs/example/component_allowmixing.puml

Время: парсинг 0.674с | proto 0.194с | диаграмма 0.017с | всего 0.885с

--- Начало PlantUML ---
@startuml
allowmixing
hide empty members
--- ... ---


# README

In [54]:
import re
from pathlib import Path
from typing import Any, Dict, List, Tuple, Optional, Set
from collections import defaultdict


# ==== безопасные геттеры/итераторы ====
def _get(obj: Any, attr: str, default=None):
    if isinstance(obj, dict):
        return obj.get(attr, default)
    return getattr(obj, attr, default)


def _iter(obj: Any, attr: str) -> List[Any]:
    v = _get(obj, attr, None)
    return list(v) if v else []


# ==== утилиты типов/сигнатур ====
def _short_typename(t: Any) -> str:
    s = str(t)
    s = re.sub(r'`[^`]*`', '', s)
    s = s.replace('...', '')
    s = re.sub(r'\bchan\b<?-?>?', ' ', s)
    s = re.sub(r'\bmap\s*$begin:math:display$[^$end:math:display$]*\]', ' ', s)
    s = re.sub(r'<[^>]*>', ' ', s)
    s = re.sub(r'[*$begin:math:display$$end:math:display$$begin:math:text$$end:math:text$]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    parts = re.split(r'[/\s\.]', s)
    parts = [p for p in parts if p]
    return parts[-1] if parts else s


def _classify_import(imp_path: str) -> str:
    first = imp_path.split("/")[0]
    return "stdlib" if "." not in first else "external"


# ==== агрегация по директориям ====
def _group_files_by_dir(structs_map: Dict[str, Any]) -> Dict[str, List[str]]:
    dir_to_files: Dict[str, List[str]] = defaultdict(list)
    for rel_path in structs_map.keys():
        p = Path(rel_path)
        d = str(p.parent).replace("\\", "/")
        dir_to_files[d].append(p.name)
    return dir_to_files


# ==== извлечение внешних импортов ====
def _collect_external_imports(structs_map: Dict[str, Any]) -> Dict[str, Dict[str, Set[str]]]:
    """
    Возвращает: { external_import : { dir_path : set(files) } }
    """
    usage: Dict[str, Dict[str, Set[str]]] = defaultdict(lambda: defaultdict(set))
    for rel_path, meta in structs_map.items():
        p = Path(rel_path)
        d = str(p.parent).replace("\\", "/")
        for imp in _iter(meta, "imports"):
            path = _get(imp, "path", "")
            if not path:
                continue
            if _classify_import(path) == "external":
                usage[path][d].add(p.name)
    return usage


# ==== извлечение тестов ====
def _collect_tests(structs_map: Dict[str, Any]) -> Dict[str, Dict[str, List[Tuple[str, int]]]]:
    """
    Возвращает: { dir_path : { file_name : [(test_name, line), ...] } }
    Ищем в *_test.go функции: Test*, Benchmark*, Example*
    """
    tests_by_dir: Dict[str, Dict[str, List[Tuple[str, int]]]] = defaultdict(lambda: defaultdict(list))
    for rel_path, meta in structs_map.items():
        p = Path(rel_path)
        if not p.name.endswith("_test.go"):
            continue
        d = str(p.parent).replace("\\", "/")
        for f in _iter(meta, "funcs"):
            name = _get(f, "name", "")
            if not name:
                continue
            if name.startswith(("Test", "Benchmark", "Example")):
                line = _get(f, "line", None) or 1
                tests_by_dir[d][p.name].append((name, line))
    # сортировка тестов в каждом файле
    for d in tests_by_dir:
        for fn in tests_by_dir[d]:
            tests_by_dir[d][fn].sort(key=lambda x: (x[1], x[0]))
    return tests_by_dir


# ==== генерация ссылок ====
def _link_to(rel_path: str, line: Optional[int] = None, repo_base_url: Optional[str] = None) -> str:
    """
    Строит Markdown-ссылку на файл (и строку) в виде:
      - относительный: ./path#L123
      - репозиторий:    {repo_base_url}/path#L123
    repo_base_url — опционально, например: https://github.com/org/repo/blob/branch
    """
    href = f"{rel_path}"
    if repo_base_url:
        href = f"{repo_base_url.rstrip('/')}/{rel_path}"
    if line:
        href = f"{href}#L{line}"
    return href


def _param_sig(params: List[Any]) -> str:
    out = []
    for p in params or []:
        nm = _get(p, "name", None)
        tp = _short_typename(_get(p, "type", ""))
        out.append(f"{nm}: {tp}" if nm else tp)
    return ", ".join(out)


def _results_sig(results: List[Any]) -> str:
    if not results:
        return ""
    return " → " + ", ".join(_short_typename(r) for r in results)


def _collect_known_structs(structs_map: Dict[str, Any]) -> Set[str]:
    names: Set[str] = set()
    for meta in structs_map.values():
        for t in _iter(meta, "types"):
            if _get(t, "kind") == "struct":
                nm = _get(t, "name")
                if nm:
                    names.add(nm)
    return names


def _func_internal_struct_usage(f: Any, known_structs: Set[str]) -> Tuple[Set[str], Set[str]]:
    in_set, out_set = set(), set()
    for p in _iter(f, "params"):
        base = _short_typename(_get(p, "type", ""))
        if base in known_structs:
            in_set.add(base)
    for r in _get(f, "results", []) or []:
        base = _short_typename(r)
        if base in known_structs:
            out_set.add(base)
    return in_set, out_set


def _example_call(meta_pkg: Optional[str], f: Any, in_structs: Set[str], out_structs: Set[str]) -> str:
    name = _get(f, "name", "Func")
    recv = _get(f, "receiver")
    params = _iter(f, "params")
    results = _get(f, "results", []) or []

    decl_lines: List[str] = []
    used_vars: List[str] = []

    if recv:
        recv_t = _short_typename(recv)
        decl_lines.append(f"var recv {recv_t}")
    for i, p in enumerate(params or []):
        tp = _short_typename(_get(p, "type", ""))
        nm = _get(p, "name", None) or f"arg{i + 1}"
        used_vars.append(nm)
        decl_lines.append(f"var {nm} {tp}")

    lhs = ""
    if results:
        outs = [f"res{i + 1}" for i in range(len(results))]
        lhs = ", ".join(outs) + " := "

    call_args = ", ".join(used_vars)
    call = f"recv.{name}({call_args})" if recv else f"{name}({call_args})"

    lines = ["```go"]
    if decl_lines:
        lines.extend(decl_lines)
        lines.append("")
    lines.append(f"{lhs}{call}")
    lines.append("```")
    return "\n".join(lines)


def _classify_import(imp_path: str) -> str:
    first = imp_path.split("/")[0]
    return "stdlib" if "." not in first else "external"


def _index_packages(structs_map: Dict[str, Any]) -> Dict[str, Set[str]]:
    pkg_to_dirs = defaultdict(set)
    for rel_path, meta in structs_map.items():
        pkg = _get(meta, "package", None)
        if pkg:
            pkg_to_dirs[pkg].add(str(Path(rel_path).parent).replace("\\", "/"))
    return pkg_to_dirs


def _guess_internal(import_path: str, pkg_to_dirs: Dict[str, Set[str]]) -> Optional[Tuple[str, Set[str]]]:
    last = import_path.split("/")[-1]
    if last in pkg_to_dirs:
        return last, pkg_to_dirs[last]
    return None


# ────────── генератор README с новой структурой ──────────
def build_readmes_by_dir_ru_v2(structs_map: Dict[str, Any]) -> Dict[str, str]:
    """
    Формирует README по директориям в формате:
      #  <dir or pkg/dir>
      ## <file.go>
      ### Структуры
      ### Функции
      ### Импорты
    """
    known_structs = _collect_known_structs(structs_map)
    pkg_to_dirs = _index_packages(structs_map)

    # группировка по директориям
    dir_to_files: Dict[str, List[str]] = defaultdict(list)
    for rel_path in structs_map.keys():
        p = Path(rel_path)
        d = str(p.parent).replace("\\", "/")
        dir_to_files[d].append(p.name)

    # сводка по каждому файлу
    file_summaries: Dict[str, Dict[str, Any]] = {}
    for rel_path, meta in structs_map.items():
        pkg = _get(meta, "package", None)
        types = _iter(meta, "types")
        funcs = _iter(meta, "funcs")
        imps = _iter(meta, "imports")

        summary = {
            "package": pkg,
            "structs": [],  # (name, line)
            "interfaces": [],  # (name, line)
            "aliases": [],  # (name, line)
            "methods": [],  # (name, recv, in_sig, out_sig, line, doc, in_set, out_set)
            "funcs": [],  # (name, in_sig, out_sig, line, doc, in_set, out_set)
            "imports": {"internal": [], "stdlib": [], "external": []},
            "raw_funcs": funcs,  # сохраним для примера вызова
        }

        for t in types:
            kind = _get(t, "kind", "")
            name = _get(t, "name", "")
            line = _get(t, "line", None)
            if kind == "struct":
                summary["structs"].append((name, line))
            elif kind == "interface":
                summary["interfaces"].append((name, line))
            elif kind == "alias":
                summary["aliases"].append((name, line))

        for f in funcs:
            name = _get(f, "name", "")
            line = _get(f, "line", None)
            params = _iter(f, "params")
            results = _get(f, "results", []) or []
            doc = _get(f, "doc", None)
            recv = _get(f, "receiver", None)
            in_sig = _param_sig(params)
            out_sig = _results_sig(results)
            in_set, out_set = _func_internal_struct_usage(f, known_structs)
            if recv:
                summary["methods"].append((name, _short_typename(recv), in_sig, out_sig, line, doc, in_set, out_set))
            else:
                summary["funcs"].append((name, in_sig, out_sig, line, doc, in_set, out_set))

        for imp in imps:
            path = _get(imp, "path", "")
            guess = _guess_internal(path, pkg_to_dirs)
            if guess:
                pkg_name, dirs = guess
                summary["imports"]["internal"].append((path, pkg_name, sorted(dirs)))
            else:
                cls = _classify_import(path)
                summary["imports"][cls].append(path)

        file_summaries[rel_path] = summary

    # генерация README по каждой директории
    readmes: Dict[str, str] = {}
    for d, files in sorted(dir_to_files.items()):
        # заголовок: предпочитаем pkg/dir если в каталоге один пакет
        pkgs = sorted({file_summaries[(f"{d}/{fn}" if d != "." else fn)]["package"]
                       for fn in files if (f"{d}/{fn}" if d != "." else fn) in file_summaries and
                       file_summaries[(f"{d}/{fn}" if d != "." else fn)]["package"]})
        title_prefix = pkgs[0] if len(pkgs) == 1 and pkgs[0] else ""
        header = f"{title_prefix}/{d}" if title_prefix else d
        header = "/" if header in ("", ".") else header

        lines: List[str] = [f"#  {header}", ""]

        # по каждому файлу в этой директории
        for fn in sorted(files):
            rel_file = f"{d}/{fn}" if d != "." else fn
            s = file_summaries.get(rel_file)
            if not s:
                # файл без разбора — просто перечислим
                lines.append(f"## {fn}")
                lines.append("")
                lines.append("*(Нет данных анализа)*")
                lines.append("")
                continue

            lines.append(f"## {fn}")

            # локальный хелпер для ссылок
            def _link(line_no: Optional[int], text: str) -> str:
                if line_no:
                    return f"[{text}](./{rel_file}#L{line_no})"
                return f"[{text}](./{rel_file})"

            # Структуры
            lines.append("### Структуры")
            if s["structs"] or s["interfaces"] or s["aliases"]:
                for name, line_no in sorted(s["structs"], key=lambda x: (x[1] or 0, x[0])):
                    lines.append(f"- {_link(line_no, name)}")
                for name, line_no in sorted(s["interfaces"], key=lambda x: (x[1] or 0, x[0])):
                    lines.append(f"- {_link(line_no, name)} *(interface)*")
                for name, line_no in sorted(s["aliases"], key=lambda x: (x[1] or 0, x[0])):
                    lines.append(f"- {_link(line_no, name)} *(alias)*")
            else:
                lines.append("- —")
            lines.append("")

            # Функции (методы + свободные)
            lines.append("### Функции")
            any_funcs = False

            # Методы
            for name, recv, in_sig, out_sig, line_no, doc, in_set, out_set in sorted(
                    s["methods"], key=lambda x: (x[4] or 0, x[0])
            ):
                any_funcs = True
                sig = f"({recv}) {name}({in_sig}){out_sig}"
                lines.append(f"- {_link(line_no, sig)}")
                if doc:
                    lines.append(f"  - Комментарий: {doc.strip().splitlines()[0]}")
                if in_set or out_set:
                    # найти исходный объект f для корректного примера
                    fmeta = None
                    for f in _iter(structs_map[rel_file], "funcs"):
                        if _get(f, "name", "") == name and _get(f, "receiver", None):
                            fmeta = f;
                            break
                    if fmeta:
                        lines.append("")
                        lines.append("  Пример:")
                        lines.append(_example_call(s["package"], fmeta, in_set, out_set))

            # Свободные функции
            for name, in_sig, out_sig, line_no, doc, in_set, out_set in sorted(
                    s["funcs"], key=lambda x: (x[3] or 0, x[0])
            ):
                any_funcs = True
                sig = f"{name}({in_sig}){out_sig}"
                lines.append(f"- {_link(line_no, sig)}")
                if doc:
                    lines.append(f"  - Комментарий: {doc.strip().splitlines()[0]}")
                if in_set or out_set:
                    fmeta = None
                    for f in _iter(structs_map[rel_file], "funcs"):
                        if _get(f, "name", "") == name and not _get(f, "receiver", None):
                            fmeta = f;
                            break
                    if fmeta:
                        lines.append("")
                        lines.append("  Пример:")
                        lines.append(_example_call(s["package"], fmeta, in_set, out_set))

            if not any_funcs:
                lines.append("- —")
            lines.append("")

            # Импорты
            lines.append("### Импорты")
            any_imp = False
            if s["imports"]["internal"]:
                any_imp = True
                lines.append("- **Внутренние**:")
                for imp_path, pkg_name, dirs in s["imports"]["internal"]:
                    where = ", ".join(f"`{p}`" for p in dirs) if dirs else "не найдено"
                    lines.append(f"  - `{imp_path}` → пакет `{pkg_name}` в {where}")
            if s["imports"]["stdlib"]:
                any_imp = True
                lines.append("- **Стандартная библиотека**:")
                for imp in s["imports"]["stdlib"]:
                    lines.append(f"  - `{imp}`")
            if s["imports"]["external"]:
                any_imp = True
                lines.append("- **Внешние**:")
                for imp in s["imports"]["external"]:
                    lines.append(f"  - `{imp}`")
            if not any_imp:
                lines.append("- —")
            lines.append("")

        readmes[d] = "\n".join(lines).rstrip() + "\n"

    return readmes

In [165]:
# ==== ГЛАВНЫЙ README ====
def build_root_readme(structs_map: Dict[str, Any],
                      repo_root: str | Path = ".",
                      repo_base_url: Optional[str] = None,
                      child_readme_name: str = "README.md") -> str:
    """
    Строит главный README:
      • Ссылки на все дочерние README (по директориям).
      • Глобальная сводка внешних импортов (во всех пакетах).
      • Глобальная сводка тестов (во всех директориях).
    repo_base_url — если задан, ссылки будут абсолютные на репозиторий; иначе относительные.
    """
    dir_to_files = _group_files_by_dir(structs_map)
    external_usage = _collect_external_imports(structs_map)
    tests = _collect_tests(structs_map)

    lines: List[str] = ["# Обзор проекта", ""]

    # 1) Индекс директорий (ссылки на дочерние README)
    lines.append("## Индекс директорий")
    if not dir_to_files:
        lines.append("- —")
    else:
        for d in sorted(dir_to_files.keys()):
            rel = "." if d == "." else d
            child_path = f"{rel}/{child_readme_name}" if rel != "." else child_readme_name
            url = _link_to(child_path, repo_base_url=repo_base_url)
            pretty = "/" if rel == "." else rel
            lines.append(f"- [{pretty}]({url})")
    lines.append("")

    # 2) Внешние импорты (глобально)
    lines.append("## Внешние импорты (по всем пакетам)")
    if not external_usage:
        lines.append("- —")
    else:
        # сгруппируем по домену (первый сегмент с точкой) для читаемости
        by_domain: Dict[str, List[str]] = defaultdict(list)
        for imp in external_usage.keys():
            domain = imp.split("/")[0]
            by_domain[domain].append(imp)
        for domain in sorted(by_domain.keys()):
            lines.append(f"### {domain}")
            for imp in sorted(by_domain[domain]):
                lines.append(f"- `{imp}`")
                # где используется
                for d in sorted(external_usage[imp].keys()):
                    files = sorted(external_usage[imp][d])
                    if d == ".":
                        d_disp = "/"
                    else:
                        d_disp = d
                    # ссылки на файлы
                    file_links = []
                    for fn in files:
                        rel_path = f"{d}/{fn}" if d != "." else fn
                        file_links.append(f"[`{fn}`]({_link_to(rel_path, repo_base_url=repo_base_url)})")
                    lines.append(f"  - {d_disp}: {', '.join(file_links)}")
    lines.append("")

    # 3) Тесты (глобально)
    lines.append("## Тесты (по всем пакетам)")
    if not tests:
        lines.append("- —")
    else:
        total_tests = 0
        for d in sorted(tests.keys()):
            lines.append(f"### {('/' if d == '.' else d)}")
            for fn in sorted(tests[d].keys()):
                rel_path = f"{d}/{fn}" if d != "." else fn
                lines.append(f"- `{fn}`")
                for name, line in tests[d][fn]:
                    total_tests += 1
                    link = _link_to(rel_path, line, repo_base_url=repo_base_url)
                    lines.append(f"  - [{name}]({link})")
        lines.append("")
        lines.append(f"**Итого тестов:** {total_tests}")
    lines.append("")

    return "\n".join(lines).rstrip() + "\n"


# ==== запись главного README на диск (опционально) ====
def write_root_readme(base_dir: str | Path, content: str, filename: str = "README.md") -> Path:
    base = Path(base_dir)
    base.mkdir(parents=True, exist_ok=True)
    out = base / filename
    out.write_text(content, encoding="utf-8")
    return out

In [166]:
# structures = make_structures(PROJECT_PATH, IGNORE_DIRS)

readmes = build_readmes_by_dir_ru_v2(structs)

# посмотреть конкретный README:
print(readmes.get("/home/jovyan/work/tree_docs/example/", ""))  # корневая директория
# или сохранить на диск рядом с проектом:
from pathlib import Path

base = Path(PROJECT_PATH)
for rel_dir, md in readmes.items():
    out = (base / rel_dir) / "README.md"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(md, encoding="utf-8")

In [168]:
root_md = build_root_readme(
    structs,
    repo_root=PROJECT_PATH,
    repo_base_url=None  # или "https://github.com/org/repo/blob/main" для абсолютных ссылок
)

write_root_readme(PROJECT_PATH, root_md)  # при необходимости записать в корень

PosixPath('/home/jovyan/work/tree_docs/example/k8sgpt/README.md')